# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Soham334/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** regularized logistic regression (`class_weight="balanced"`, `max_iter=2000`, seed 42), with
standardized numeric features and one-hot encoded categorical features — the same model, same hyperparameters,
and same feature allowlist as `work/notebooks/capstone.ipynb`.

It fits this lane for a few concrete reasons:

- **A transparent baseline model.** Logistic regression is the smallest model that's defensible for a yes/no
  observed label. It doesn't try to be the strongest possible score; it tries to be a score a reviewer can audit.
- **Interpretable coefficients.** Each fitted coefficient reads directly as "this signal pushes the score up or
  down," which a decision tree ensemble or gradient-boosted model would not offer as directly.
- **Probability scores suitable for ranking.** `predict_proba` gives a continuous score per page, which is what a
  review queue actually needs (rank by decline risk), not just a class label.
- **A simple, apples-to-apples comparison against the Week-4 rule-based baseline.** Both the model and the
  `stale_visible` rule are scored with the identical target, identical rows, and identical metric functions, so
  the comparison in Section 3 is honest rather than mixing two different evaluation setups.

This is an **observational, decision-support ranking exercise** — the model's output is a prioritized human-review
queue, not a forecast of future traffic, not a causal claim about what drives decline, and not a claim about how
Google's ranking algorithm works.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "scripts"))
from ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES,
    NUMERIC_COLUMNS,
    CATEGORICAL_COLUMNS,
    BOOLEAN_COLUMNS,
    to_bool_series,
)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
np.random.seed(SEED)

REPO_ROOT = Path.cwd().parent.parent
DATA_PATH = REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Starter dataset not found at data/raw/content_refresh_anonymized.csv. "
        "Restore it from git before running this notebook."
    )

df_raw = pd.read_csv(DATA_PATH)
print("Raw rows:", len(df_raw), "| Raw columns:", len(df_raw.columns))

# Same cleaning + population filter as capstone.ipynb / scripts/01_prepare_features.py
for column in NUMERIC_COLUMNS:
    df_raw[column] = pd.to_numeric(df_raw[column], errors="coerce") if column in df_raw.columns else 0
for column in BOOLEAN_COLUMNS:
    df_raw[column] = to_bool_series(df_raw[column]) if column in df_raw.columns else False
for column in CATEGORICAL_COLUMNS:
    if column in df_raw.columns:
        df_raw[column] = df_raw[column].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
    else:
        df_raw[column] = "unknown"

numeric_fill_zero = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "impressions_last_30d",
    "clicks_last_30d", "sessions_last_30d", "impressions_prev_30d",
    "clicks_prev_30d", "sessions_prev_30d", "content_age_days", "age_tier_order",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "trend_pct",
]
for column in numeric_fill_zero:
    df_raw[column] = df_raw[column].replace([np.inf, -np.inf], np.nan).fillna(0)

df = df_raw[(df_raw["impressions_90d"] > 0) & (df_raw["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
print(f"Modeling population: {len(df):,} rows (from {len(df_raw):,} raw rows).")

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

# Leakage guard: label-source fields and pseudonymous IDs must never be in the model allowlist
assert not (set(MODEL_NUMERIC_FEATURES) | set(MODEL_CATEGORICAL_FEATURES)) & {"trend_direction", "trend_pct", "content_id", "client_id"}, \
    "Leakage guard failed."
print("Leakage guard passed: no label-derived field or ID in the model feature lists.")
print(f"Numeric features ({len(MODEL_NUMERIC_FEATURES)}):", MODEL_NUMERIC_FEATURES)
print(f"Categorical features ({len(MODEL_CATEGORICAL_FEATURES)}):", MODEL_CATEGORICAL_FEATURES)
print("Base rate (is_declining_label = 1):", round(df["is_declining_label"].mean(), 4))

Raw rows: 30000 | Raw columns: 44
Modeling population: 30,000 rows (from 30,000 raw rows).


Leakage guard passed: no label-derived field or ID in the model feature lists.
Numeric features (18): ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical features (8): ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
Base rate (is_declining_label = 1): 0.5421


## 2. Split design

**GroupKFold(n_splits=5)**, grouped by `client_id` — the same split used in `capstone.ipynb`, not a random row
split.

Grouping by client matters because rows from the same client are not independent — pages belonging to one client
tend to share a lot of the same underlying visibility and content patterns. If the same client's pages could land
in both the training fold and the test fold, the model could partly "recognize" that client rather than genuinely
generalizing to a page it has never seen anything from. `GroupKFold` guarantees that every client's rows sit
entirely in one fold, so a client never leaks across the train/test boundary.

A single random 80/20 grouped split was tried first (in the capstone) and rejected: the baseline-flagged rows are
concentrated in only 10 of 32 clients, so one random grouped split has a real chance of putting zero
baseline-flagged clients in the test fold — which is exactly what happened. Five-fold grouped CV, evaluated on
concatenated out-of-fold (OOF) predictions across all rows, is the smallest defensible fix: every client, and
therefore every baseline-flagged row, ends up in exactly one out-of-fold evaluation.

In [2]:
# Week-4 baseline rule, reused exactly (not redefined here): stale + visible -> REFRESH_REVIEW
df["baseline_stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["baseline_visible"] = (df["impressions_90d"] > 0).astype(int)
df["baseline_flag"] = (df["baseline_stale"] & df["baseline_visible"]).astype(int)
print("Baseline REFRESH_REVIEW flags:", df["baseline_flag"].sum(), "of", len(df),
      f"({df['baseline_flag'].mean():.2%})")

X = df[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].astype(str)

gkf = GroupKFold(n_splits=5)
fold_rows = []
for fold_i, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
    train_clients = set(groups.iloc[train_idx])
    test_clients = set(groups.iloc[test_idx])
    overlap = train_clients & test_clients
    fold_rows.append({
        "fold": fold_i,
        "n_train_rows": int(len(train_idx)),
        "n_test_rows": int(len(test_idx)),
        "n_test_clients": int(len(test_clients)),
        "client_overlap_train_test": len(overlap),  # must always be 0
    })

fold_check = pd.DataFrame(fold_rows)
print(fold_check.to_string(index=False))
assert (fold_check["client_overlap_train_test"] == 0).all(), "A client leaked across train/test in some fold."
print("\nConfirmed: no client_id appears in both the train and test side of any fold.")

Baseline REFRESH_REVIEW flags: 174 of 30000 (0.58%)
 fold  n_train_rows  n_test_rows  n_test_clients  client_overlap_train_test
    0         22992         7008               1                          0
    1         24269         5731               7                          0
    2         24247         5753               8                          0
    3         24245         5755               8                          0
    4         24247         5753               8                          0

Confirmed: no client_id appears in both the train and test side of any fold.


## 3. Train + compare vs my baseline

Same dataset, same target (`is_declining_label`), same `MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES`
allowlist, same `stale_visible` baseline, same grouped 5-fold OOF evaluation as `capstone.ipynb`. The model below
is refit independently inside each fold on the training clients only, so every row's OOF score comes from a fold
that never saw that row's client during training. Nothing about the baseline rule is changed.

The table below is generated fresh from this run, not copied from `capstone_metrics.json` — the numbers should
(and do) match, since it's the identical pipeline on the identical data.

In [3]:
import json
oof_model_score = np.zeros(len(df))
oof_fold_id = np.full(len(df), -1)

for fold_i, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train = y.iloc[train_idx]

    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    prep = ColumnTransformer([("num", num_pipe, MODEL_NUMERIC_FEATURES), ("cat", cat_pipe, MODEL_CATEGORICAL_FEATURES)])
    fold_model = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))])
    fold_model.fit(X_train, y_train)

    scores = fold_model.predict_proba(X_test)[:, 1]
    oof_model_score[test_idx] = scores
    oof_fold_id[test_idx] = fold_i

df["oof_model_score"] = oof_model_score
df["oof_fold"] = oof_fold_id

def p_at_k(y_true, scores, k, seed=SEED):
    frame = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)})
    frame = frame.sample(frac=1.0, random_state=seed)
    top = frame.sort_values("score", ascending=False, kind="mergesort").head(min(k, len(frame)))
    return float(top["y"].mean())

y_all = df["is_declining_label"].values
baseline_all = df["baseline_flag"].values
model_all = df["oof_model_score"].values

comparison = pd.DataFrame({
    "metric": ["ROC-AUC", "Average precision", "Precision@20", "Precision@50", "Precision@100"],
    "baseline (stale_visible)": [
        roc_auc_score(y_all, baseline_all),
        average_precision_score(y_all, baseline_all),
        p_at_k(y_all, baseline_all, 20),
        p_at_k(y_all, baseline_all, 50),
        p_at_k(y_all, baseline_all, 100),
    ],
    "model (logistic regression)": [
        roc_auc_score(y_all, model_all),
        average_precision_score(y_all, model_all),
        p_at_k(y_all, model_all, 20),
        p_at_k(y_all, model_all, 50),
        p_at_k(y_all, model_all, 100),
    ],
})
comparison["baseline (stale_visible)"] = comparison["baseline (stale_visible)"].round(3)
comparison["model (logistic regression)"] = comparison["model (logistic regression)"].round(3)
print(comparison.to_string(index=False))

# Cross-check against the committed capstone receipt (does not overwrite it)
capstone_metrics = json.loads((REPO_ROOT / "work" / "outputs" / "capstone_metrics.json").read_text())
checks = {
    "model_roc_auc": roc_auc_score(y_all, model_all),
    "baseline_roc_auc": roc_auc_score(y_all, baseline_all),
    "model_average_precision": average_precision_score(y_all, model_all),
    "baseline_average_precision": average_precision_score(y_all, baseline_all),
}
print("\nCross-check vs work/outputs/capstone_metrics.json:")
for key, value in checks.items():
    ref = capstone_metrics[key]
    print(f"  {key}: this run={value:.6f}  capstone receipt={ref:.6f}  match={abs(value - ref) < 1e-9}")

           metric  baseline (stale_visible)  model (logistic regression)
          ROC-AUC                     0.499                        0.665
Average precision                     0.542                        0.668
     Precision@20                     0.550                        0.750
     Precision@50                     0.460                        0.740
    Precision@100                     0.490                        0.810

Cross-check vs work/outputs/capstone_metrics.json:
  model_roc_auc: this run=0.664929  capstone receipt=0.664929  match=True
  baseline_roc_auc: this run=0.499173  capstone receipt=0.499173  match=True
  model_average_precision: this run=0.667819  capstone receipt=0.667819  match=True
  baseline_average_precision: this run=0.541710  capstone receipt=0.541710  match=True


## 4. Errors and interpretation

Using the out-of-fold scores at the default 0.5 threshold and the full-population coefficient refit (same
approach as `capstone.ipynb` Sections 6 and the error analysis) — nothing here changes the model, the baseline, or
the committed capstone artifacts; it re-derives the same view from the same run above.

**What the model leans on (directionally, not causally):** higher recent visibility (`log_impressions_90d`) is
associated with a *higher* predicted decline probability, while landing in a strong position or impression tier
is associated with a *lower* one, and higher recent clicks read as protective. Nothing in the coefficient list has
the near-perfect, single-feature-dominates signature that would suggest leakage.

**Where the model disagrees with the observed label:** false negatives (declining pages scored low) are the
costlier error for this use case, since they're the pages most likely to go unreviewed; false positives cost
review time on pages that turn out fine. The baseline's near-chance ROC-AUC (Section 3) is itself a negative
finding worth stating plainly — the stale+visible rule doesn't currently track this particular decline label, even
though it's a reasonable, transparent rule for its own original purpose.

Framing throughout: observed / measured / directional / decision-support. No causal claim, no forecast of future
traffic, and no claim about Google's ranking algorithm.

In [4]:
# Full-population refit for interpretation only (performance claims above use OOF scores, not this fit)
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
prep = ColumnTransformer([("num", num_pipe, MODEL_NUMERIC_FEATURES), ("cat", cat_pipe, MODEL_CATEGORICAL_FEATURES)])
final_model = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))])
final_model.fit(X, y)

names = final_model.named_steps["prep"].get_feature_names_out()
coef = final_model.named_steps["clf"].coef_[0]
effects = pd.DataFrame({"feature": names, "coefficient": coef})
effects["abs_coefficient"] = effects["coefficient"].abs()
top_effects = effects.sort_values("abs_coefficient", ascending=False).head(10)[["feature", "coefficient"]]
print("Top 10 coefficients by magnitude:")
print(top_effects.to_string(index=False))

# Error breakdown at the default 0.5 threshold, out-of-fold scores
threshold = 0.5
df["predicted"] = (df["oof_model_score"] >= threshold).astype(int)
df["error_type"] = np.select(
    [(df["predicted"] == 1) & (df["is_declining_label"] == 0),
     (df["predicted"] == 0) & (df["is_declining_label"] == 1)],
    ["false_positive", "false_negative"],
    default="correct",
)
print("\nError breakdown (OOF scores, threshold 0.5):")
print(df["error_type"].value_counts().to_string())
recall = ((df["predicted"] == 1) & (df["is_declining_label"] == 1)).sum() / df["is_declining_label"].sum()
precision = ((df["predicted"] == 1) & (df["is_declining_label"] == 1)).sum() / max(df["predicted"].sum(), 1)
print(f"Recall {recall:.3f}, precision {precision:.3f}, flagged rate {df['predicted'].mean():.3f}")

# Reason-code mix (top-20% model score vs baseline flag) - same rule as capstone.ipynb, not written to disk here
cut = df["oof_model_score"].quantile(0.80)
conditions = [
    (df["oof_model_score"] >= cut) & (df["baseline_flag"] == 1),
    (df["oof_model_score"] >= cut) & (df["baseline_flag"] == 0),
    (df["oof_model_score"] < cut) & (df["baseline_flag"] == 1),
]
choices = ["MODEL_HIGH_AND_STALE_VISIBLE", "MODEL_HIGH", "STALE_VISIBLE"]
df["reason_code"] = np.select(conditions, choices, default="REVIEW_IF_NEEDED")
print("\nReason-code mix (matches capstone's ranked-queue logic):")
print(df["reason_code"].value_counts().to_string())

Top 10 coefficients by magnitude:
                          feature  coefficient
         num__log_impressions_90d     1.409672
         cat__position_tier_top_3    -0.737107
         cat__impression_tier_low     0.691493
   cat__impression_tier_excellent    -0.655116
              num__log_clicks_90d    -0.627646
                  num__word_count     0.539691
   cat__word_count_tier_1000-2000     0.432177
        cat__freshness_tier_31-90    -0.389194
cat__content_type_keyword article     0.380934
                num__avg_position    -0.372445

Error breakdown (OOF scores, threshold 0.5):
error_type
correct           18857
false_positive     6729
false_negative     4414
Recall 0.729, precision 0.638, flagged rate 0.619

Reason-code mix (matches capstone's ranked-queue logic):
reason_code
REVIEW_IF_NEEDED                23857
MODEL_HIGH                       5969
STALE_VISIBLE                     143
MODEL_HIGH_AND_STALE_VISIBLE       31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.